# AI Application Development with Python

# 1. Calling LLM APIs from Python

## What is LLM API

An LLM API is a way for a Python program to ask a language model a question and get its answer back. Your application sends the prompt over the internet, and the provider runs the model for you. You can then use the response in a website, chatbot, script, or backend service.

For example, a support application could send a customer's question to a model and show the answer on the screen.

## Why it is useful

- You can add AI features without training a model yourself.
- The model runs on the provider's infrastructure, so you do not need a powerful computer to host it.
- The same approach can support chat, summarisation, text extraction, and many other features.
- It is quick to test an idea before building a larger application.

## Flowchart

![AI architecture](https://media.licdn.com/dms/image/v2/D4E12AQE5X1E70ZZFfA/article-inline_image-shrink_400_744/article-inline_image-shrink_400_744/0/1692395482436?e=2147483647&v=beta&t=eczkLxRIwNtqKnc05UCMpHPBKLtUuxypKhJ_sOF_x3Q
)

## Program

### Step 1: Install libraries

In [ ]:
pip install groq python-dotenv

### Step 2: Configure the API key

Create a `.env` file beside this notebook:

```env
GROQ_API_KEY=your_actual_groq_api_key
```


### Step 3: Call the API

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
print(f"Using GROQ_API_KEY: {api_key}")

if not api_key:
    raise ValueError("GROQ_API_KEY is not configured")

client = Groq(api_key=api_key)

question = input("Enter your question: ").strip()
if not question:
    raise ValueError("Please enter a question.")

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a helpful Python trainer. Explain clearly."},
        {"role": "user", "content": question},
    ],
)

print(response.choices[0].message.content)


# 2. API Keys, Environment Variables, and Secret Management

## What this means

- An **API key** is like a password that lets your application access a service. 
- An **environment variable** is a setting that your program reads when it runs. 
- **Secret management** is the broader practice of keeping sensitive values safe and controlling who can use them.

For a local project, you might put the key in a `.env` file and read it from Python. In a deployed application, you would normally use the hosting platform's secret settings or a dedicated secret manager.

## Why it is useful

- Your API key does not need to be written directly into your Python code.
- You can use different keys and settings for local development and production.
- It is easier to update a key without editing the application.
- Proper secret storage reduces the chance of exposing credentials in Git or logs.

## Program

Create a `.env` file:

```env
GROQ_API_KEY=your_actual_groq_api_key
APP_ENV=development
```

Add `.env` to `.gitignore`:

```gitignore
.env
```


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
app_env = os.getenv("APP_ENV", "development")

if not api_key:
    raise RuntimeError("GROQ_API_KEY is not configured")

print("Environment:", app_env)
print("API key configured:", bool(api_key))
# Do not print the key itself.


In [ ]:
%pip install groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

result = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Give a one-sentence definition of an SDK."}],
)

print(result.choices[0].message.content)


# 3. Prompt Templates and Reusable AI Functions

## Definition:

- A **prompt** is the instruction or question you send to an AI model. 
- A **prompt template** is a reusable version of that instruction, with placeholders for values that change.

For example, instead of writing a new prompt for every programming topic, you can keep one template with a `{topic}` placeholder and fill it with `FastAPI`, `decorators`, or another topic.

## Why it is useful

- You can reuse the same instructions with different input values.
- Responses are more consistent because the core prompt stays the same.
- Prompt wording is easier to update in one place.
- Separating prompt construction from API calls keeps the code easier to maintain.

## Program

Install LangChain's core package for its prompt-template utility:


In [ ]:
%pip install langchain-core groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq
from langchain_core.prompts import PromptTemplate

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

prompt_template = PromptTemplate.from_template(
    "Explain {topic} in simple language. "
    "Include a short Python example and one use case."
)

def ask_ai(prompt: str) -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": "You are a clear and practical Python trainer."},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

def explain_topic(topic: str) -> str:
    prompt = prompt_template.format(topic=topic)
    return ask_ai(prompt)

print(explain_topic("Python generators"))


# 4. Chat History and Conversation Context

## What this means

- **Chat history** is the earlier messages in a conversation. 
- **Conversation context** is the information the model receives to understand the current message.

If a user asks, “What is FastAPI?” and then asks, “What are its advantages?”, the application usually needs to send the earlier exchange along with the follow-up. The model cannot reliably refer to messages that the application has not provided or saved through a supported conversation feature.

## Why it is useful

- Users can ask follow-up questions without repeating everything.
- The application can reuse relevant details from earlier messages.
- Summarising or limiting old messages can help control request size as a conversation grows.


## Program



In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

messages = [
    {"role": "system", "content": "You are a helpful Python tutor. Use clear, concise explanations."}
]

def chat(question: str) -> str:
    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
    )
    answer = response.choices[0].message.content
    messages.append({"role": "assistant", "content": answer})
    return answer

print(chat("What is FastAPI?")[:50])  #This result gives large output, so we are printing only first 50 characters of the answer.
print("first answer ended---->")
print("\nFollow-up:")
print(chat("What are its advantages?"))


# 5. Function Calling and Tool Calling

## Definitions:
- Function calling or tool calling allows an AI model to use Python functions to perform tasks instead of only generating text.
- For example, if a user asks, "What is the status of my application?", the AI model can decide to use a Python function that checks the application status.
- The model does not execute the Python function itself. It requests the function call, and the Python application executes it and returns the result to the model.



## Why it is useful
- The LLM can choose the appropriate function based on the user's question.
- The LLM can identify the required parameters and their values.
- Python functions can retrieve real information from databases or APIs.
- Python controls function execution and validates the arguments before running the function.

## Program



In [ ]:

import os
import json
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 1. Create the Python function (the tool)
def get_application_status(application_id: int) -> dict:
    statuses = {
        501: "Interview Scheduled",
        502: "Under Review"
    }

    return {
        "application_id": application_id,
        "status": statuses.get(application_id, "Not Found")
    }


# 2. Describe the tool to the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_application_status",
            "description": "Get the status of a job application.",
            "parameters": {
                "type": "object",
                "properties": {
                    "application_id": {
                        "type": "integer",
                        "description": "The ID of the application"
                    }
                },
                "required": ["application_id"],
                "additionalProperties": False
            }
        }
    }
]


# 3. Ask the LLM a question
messages = [
    {
        "role": "user",
        "content": "What is the status of application 501?"
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

# 4. Check whether the LLM requested a tool
assistant_message = response.choices[0].message
messages.append(assistant_message)

if assistant_message.tool_calls:

    for tool_call in assistant_message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        # 5. Execute the requested Python function
        if function_name == "get_application_status":

            application_id = arguments.get("application_id")

            if (
                not isinstance(application_id, int)
                or isinstance(application_id, bool)
            ):
                raise ValueError("application_id must be an integer")

            result = get_application_status(application_id)

        else:
            raise ValueError("Unknown tool requested")

        print("Tool result:", result)

        # 6. Return the result to the LLM
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

    # 7. Ask the LLM to generate the final answer
    final_response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages
    )

    print("Final answer:", final_response.choices[0].message.content)

else:
    print("Final answer:", assistant_message.content)


# 6. Embedding Generation and Vector Databases

## What this means

- An **embedding** turns text into a list of numbers that captures some of its meaning. Texts with similar meanings often have vectors that are close to one another, even when they use different words.

- A **vector database** or vector index stores these vectors and helps find similar items. In this notebook, Hugging Face creates the embeddings and FAISS searches the vectors. This is useful when a normal keyword search is not enough.

## Why it is useful

- Search can find related wording, not just exact keyword matches.
- It can help retrieve relevant passages from documents.
- The same approach can support semantic search and recommendations.
- Hugging Face offers embedding models, and FAISS can run locally without a separate hosted database.

## Flowchart

```text
+-------------------------+
| Source text documents   |
+------------+------------+
             |
             v
+-------------------------+
| Hugging Face embedding  |
| model creates vectors   |
+------------+------------+
             |
             v
+-------------------------+
| Store vectors in FAISS  |
+-------------------------+
             |
             |       User query
             |           |
             |           v
             |   +-------------------+
             |   | Embed query       |
             |   +---------+---------+
             |             |
             v             v
       +-------------------------+
       | Search similar vectors  |
       +------------+------------+
                    |
                    v
       +-------------------------+
       | Return relevant documents|
       +-------------------------+
```

## Program



In [ ]:
pip install -U langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

documents = [
    Document(page_content="FastAPI is a Python framework for building APIs.",
             metadata={"topic": "FastAPI"}),
    Document(page_content="SQLAlchemy is a toolkit and ORM for relational databases.",
             metadata={"topic": "SQLAlchemy"}),
    Document(page_content="RAG retrieves relevant information before an LLM generates an answer.",
             metadata={"topic": "RAG"}),
]

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(documents, embedding_model)

query = "what is embeddings?"
results = vector_store.similarity_search(query, k=1)

for doc in results:
    print(doc.metadata, "->", doc.page_content)


# 7. Retrieval-Augmented Generation (RAG)

## RAG:

**Retrieval-Augmented Generation (RAG)** gives an AI model relevant information from your own documents when it answers a question. The application first searches for useful passages, then sends those passages along with the question to the model.

For example, if you ask about a company policy, the application can retrieve the matching policy section instead of relying only on the model's general knowledge.

## Advantages:

- Answers can be based on your own documents or business information.
- Updating the document collection can update what the application retrieves, without retraining the language model.
- The application can show which documents were used as supporting context.
- RAG is often a practical way to build question-answering over files, although retrieval does not guarantee that every answer will be correct.

## Flowchart

```text
Knowledge preparation:
Documents -> Split into chunks -> Create embeddings -> Store in FAISS

Question answering:
User question -> Embed question -> Retrieve relevant chunks
             -> Add chunks to prompt -> Call LLM -> Answer
```

![AI architecture](https://miro.medium.com/1*3--ogs382Na1U2v3LfVVcQ.png)

## Program

This example builds a small RAG pipeline with LangChain, Hugging Face embeddings, FAISS, and a Groq chat model.


In [ ]:
%pip install --upgrade langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-groq langsmith

In [ ]:

import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not configured in the environment or .env file")

documents = [
    Document(page_content="Employees can work remotely two days per week.",
             metadata={"source": "work_policy.txt"}),
    Document(page_content="Annual leave requests must be approved by a manager.",
             metadata={"source": "leave_policy.txt"}),
    Document(page_content="Office working hours are 9 AM to 6 PM.",
             metadata={"source": "office_policy.txt"}),
]

# Added: Chunking process
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

print("Total documents:", len(documents))
print("Total chunks:", len(chunks))

# Generate embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Store chunks instead of original documents
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer using only the supplied context. If the answer is not present, "
     "say the provided documents do not contain the answer. Do not invent policy."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

def answer_with_rag(question: str) -> str:
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    sources = sorted({doc.metadata.get("source", "unknown") for doc in retrieved_docs})
    return f"{response.content}\n\nRetrieved sources: {', '.join(sources)}"

print(answer_with_rag("How often can employees work remotely?"))


# 8. Document Loading, Chunking, and Retrieval

## Definitions:

Before an application can search documents, it needs to read them, split long text into manageable pieces, and find the pieces relevant to a question.

- **Loading** means reading content from a file or another source.
- **Chunking** means splitting long text into smaller sections.
- **Retrieval** means selecting the sections that best match a question.

These steps are commonly used when preparing documents for embedding search and RAG.

## Why it is useful

- Smaller sections are easier to search than one very large document.
- Only the most relevant passages need to be sent to the model.
- Metadata can help identify which file a passage came from.
- Overlap between chunks can preserve context when an important sentence falls near a split.

## Flowchart

```text
+-----------------------+
| Load document         |
+-----------+-----------+
            |
            v
+-----------------------+
| Clean extracted text  |
+-----------+-----------+
            |
            v
+-----------------------+
| Split into chunks     |
+-----------+-----------+
            |
            v
+-----------------------+
| Embed and index chunks|
+-----------+-----------+
            |
            v
+-----------------------+
| Retrieve top matches  |
+-----------------------+
```

## Program

This example loads a text file, splits it using LangChain's recursive text splitter, and indexes the chunks in FAISS. Create `sample_policy.txt` in the notebook folder before running the code.


In [ ]:
%pip install -U langchain-text-splitters langchain-community langchain-huggingface sentence-transformers faiss-cpu

In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

file_path = Path("sample_policy.txt")
if not file_path.exists():
    file_path.write_text(
        "Employees can work remotely two days per week. "
        "Annual leave requests must be approved by a manager. "
        "Office hours are 9 AM to 6 PM. "
        "Employees should contact IT support for account access problems.",
        encoding="utf-8",
    )

text = file_path.read_text(encoding="utf-8")
source_document = Document(
    page_content=text,
    metadata={"source": file_path.name},
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
)
chunks = splitter.split_documents([source_document])

print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:\n{chunk.page_content}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)

query = "Who should I contact for account access problems?"
for result in vector_store.similarity_search(query, k=2):
    print("\nRetrieved:", result.page_content, "| Metadata:", result.metadata)


# 9. Basic AI Agents and Workflow Orchestration

## Definitions:

- An AI agent is an AI system that can decide what action to take to complete a user's request. It can use tools, retrieve information, and perform multiple steps to achieve a goal.

- Workflow orchestration means organising different tasks in the correct order and deciding which task should run next.

- For example, imagine an AI assistant that can search for jobs and explain technical concepts. When a user asks a question, the workflow decides which task to execute.

LangGraph helps build these workflows using three main concepts:

- State: Stores information that different steps need to share.

- Nodes: Individual functions that perform tasks.

- Edges: Define which step runs next.

## How It Works
- The user submits a request.
- A routing function checks the request.
- The workflow selects the appropriate task.
- The selected task processes the request and updates the shared state.
- The workflow returns the result.

## Flow Diagram

![AI architecture](https://miro.medium.com/1*au-XAKN2ZH2PNLtd-Ps_og.png)

## Program


In [ ]:
%pip install -U langgraph

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

class AssistantState(TypedDict):
    request: str
    route: str
    result: str

def route_request(state: AssistantState) -> dict:
    request = state["request"].lower()
    if "job" in request:
        return {"route": "jobs"}
    return {"route": "explain"}

def search_jobs(state: AssistantState) -> dict:
    # Demo result; replace with a secured database search.
    return {"result": "Demo: Python Developer — Chennai"}

def explain_concept(state: AssistantState) -> dict:
    # A real workflow could call an LLM here.
    return {"result": f"Explanation requested for: {state['request']}"}

graph_builder = StateGraph(AssistantState)
graph_builder.add_node("route_request", route_request)
graph_builder.add_node("search_jobs", search_jobs)
graph_builder.add_node("explain_concept", explain_concept)

graph_builder.add_edge(START, "route_request")
graph_builder.add_conditional_edges(
    "route_request",
    lambda state: state["route"],
    {"jobs": "search_jobs", "explain": "explain_concept"},
)
graph_builder.add_edge("search_jobs", END)
graph_builder.add_edge("explain_concept", END)

app_graph = graph_builder.compile()

result = app_graph.invoke({
    "request": "Find Python jobs in Chennai",
    "route": "",
    "result": "",
})
print(result["result"])


# 10. Integrating AI with FastAPI and SQLAlchemy

## What this means

- **FastAPI** lets you expose Python functionality through HTTP endpoints, so another application can send a request and receive a response. 
- **SQLAlchemy** helps Python code read and write relational database records.

Together with an LLM client, they can form a backend that accepts a question, gets an AI-generated answer, and saves the interaction in a database. The example uses SQLite so it can run locally without setting up a separate database server.

## Advantages:

- A frontend or another service can access the AI feature through an API.
- FastAPI validates request data and provides interactive API documentation.
- SQLAlchemy gives database operations a consistent Python interface.
- Keeping API routes, model calls, and database code organised makes the application easier to extend.

## Program

This is a compact learning example using SQLite and SQLAlchemy to store questions and answers. It calls Groq for the response.

In [ ]:
%pip install -U fastapi uvicorn sqlalchemy groq python-dotenv

Save the following code in a file named `main.py`. A notebook cell is not itself a running web server module.


In [ ]:
import os
from datetime import datetime, timezone

from dotenv import load_dotenv
from fastapi import FastAPI, Depends, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, String, Text, DateTime
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session, sessionmaker
from groq import Groq

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("GROQ_API_KEY is not configured")

client = Groq(api_key=api_key)

DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///./ai_training.db")
connect_args = {"check_same_thread": False} if DATABASE_URL.startswith("sqlite") else {}
engine = create_engine(DATABASE_URL, connect_args=connect_args)
SessionLocal = sessionmaker(bind=engine, autoflush=False, expire_on_commit=False)

class Base(DeclarativeBase):
    pass

class ChatRecord(Base):
    __tablename__ = "chat_records"

    id: Mapped[int] = mapped_column(primary_key=True, index=True)
    question: Mapped[str] = mapped_column(Text)
    answer: Mapped[str] = mapped_column(Text)
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), default=lambda: datetime.now(timezone.utc))

Base.metadata.create_all(engine)

app = FastAPI(title="AI Question API")

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=4000)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.post("/ask")
def ask_question(request: QuestionRequest, db: Session = Depends(get_db)):
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[
                {"role": "system", "content": "You are a helpful Python trainer. Explain clearly."},
                {"role": "user", "content": request.question},
            ],
        )
        answer = response.choices[0].message.content

        record = ChatRecord(question=request.question, answer=answer)
        db.add(record)
        db.commit()
        db.refresh(record)

        return {"id": record.id, "answer": answer}
    except Exception as exc:
        db.rollback()
        # In production, log details securely and distinguish provider from DB errors.
        raise HTTPException(status_code=502, detail="Could not complete the AI request.") from exc


### Run the API

In a terminal in the folder containing `main.py`, run:

```bash
uvicorn main:app --reload
```

Open `http://127.0.0.1:8000/docs`, choose `POST /ask`, and send:

```json
{
  "question": "Explain RAG in simple words"
}
```
